In [1]:
import pandas as pd
from transformers import pipeline

c:\Users\super\anaconda3\envs\testetcc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = "citizenlab/twitter-xlm-roberta-base-sentiment-finetunned"
sentiment_classifier = pipeline("text-classification", model=model_path, tokenizer=model_path)

Device set to use cpu


In [3]:
df = pd.read_csv("Data/olist_order_reviews_dataset.csv")

In [4]:
# Selecting only necessary columns for NLP analysis
nlp_df= df[['review_comment_message']]

In [5]:
def remove_duplicates_nlp_df(nlp_df, column_name='review_comment_message'):
    
    # Remove duplicates based on the specified column, keeping the first occurrence
    nlp_df = nlp_df.drop_duplicates(subset=[column_name], keep='first').reset_index(drop=True)
    
    # Display the total entries after removing duplicates
    print(f"Total entries after removing duplicates in '{column_name}': {nlp_df.shape[0]}")
    
    return nlp_df

# Remove duplicates from 'nlp_df' based on the 'review_comment_message' column
nlp_df = remove_duplicates_nlp_df(nlp_df, 'review_comment_message')

# Display the first few records to verify
nlp_df.head()

Total entries after removing duplicates in 'review_comment_message': 36922


,review_comment_message
0,NaN
1,Recebi bem antes do prazo estipulado.
2,Parabéns lojas lannister adorei comprar pela I...
3,aparelho eficiente. no site a marca do aparelh...
4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"


In [6]:
def clean_reviews(df):
    
    # Remove rows where 'review_comment_message' is empty
    df = df.dropna(subset=['review_comment_message']).reset_index(drop=True)

    # Remove duplicate rows
    df = df.drop_duplicates(subset=['review_comment_message'])

    return df

# Assuming 'nlp_df' is your dataframe
df_cleaned = clean_reviews(nlp_df)

# Display the first records to check
df_cleaned.head()

,review_comment_message
0,Recebi bem antes do prazo estipulado.
1,Parabéns lojas lannister adorei comprar pela I...
2,aparelho eficiente. no site a marca do aparelh...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,"Vendedor confiável, produto ok e entrega antes..."


In [7]:
nlp_df2 = nlp_df.sample(n=min(380, len(df)), random_state=42)

In [8]:
nlp_df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 380 entries, 30506 to 25753
Data columns (total 1 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_comment_message  380 non-null    object
dtypes: object(1)
memory usage: 5.9+ KB


In [13]:
def classify_sentiment(df, column_name='review_comment_message'):
    # Vectorized function to get sentiment classification
    def get_sentiment_classification(text):
        result = sentiment_classifier(text)
        label = result[0]['label']
        return label
        
    def get_sentiment_score(text):
        result = sentiment_classifier(text)
        scores = result[0]['score']
        return scores
    
    # Apply sentiment analysis using map for faster iteration
    df[f'{column_name}_sentiment'] = df[column_name].map(get_sentiment_classification)
    df[f'{column_name}_score'] = df[column_name].map(get_sentiment_score)
    return df

In [14]:
# Classify sentiment in 'nlp_df' based on the 'review_comment_message_clean' column
nlp_df2 = nlp_df2.dropna(subset=["review_comment_message"]).copy(); 
nlp_df2 = classify_sentiment(nlp_df2, 'review_comment_message')

# Display the sentiment results
nlp_df2[['review_comment_message', 'review_comment_message_sentiment']].head(25)

,review_comment_message,review_comment_message_sentiment
30506,"Atendeu super bem todas as expectativas, são b...",Positive
29795,Eu ainda não recebi o produto.,Neutral
27064,Meu produto consta como entregue mais ele aind...,Neutral
2383,"Otimo site, bons preços.",Positive
34500,Chegou rápido e tudo em ordem!,Positive
14100,"Não entraram em contato, apenas cancelaram o m...",Neutral
11794,"Loja boa , porem entrega bem demorada por um p...",Neutral
36663,"Muito bom, chegou antes do prazo",Positive
3068,Produto muito bom\r\nEntrega realizada antes d...,Positive
14638,Não entregaram os 2 itens comprados,Neutral


In [17]:
nlp_df2.tail(10)

,review_comment_message,review_comment_message_sentiment,review_comment_message_score
30014,fiquei muito satisfeita com o atendimento e a ...,Positive,0.990593
30944,O produto foi entregue até antes da data previ...,Positive,0.989044
34694,Foi entregue antes do prazo estabelecido,Neutral,0.963210
21704,"bom produto,estou sastifeito com o produto eu ...",Positive,0.989116
23999,Simplesmente amei!!!,Positive,0.992663
10359,De Boa qualidade\r\n,Positive,0.932982
15159,"muito legal, entrega no prazo e com qualidade.",Positive,0.990190
7678,"Produto entregue em antes do prazo previsto, e...",Positive,0.990829
4750,Muito satisfeito com a compra e a entrega. Pro...,Positive,0.990912
25753,Muito bom comprar no baratheon,Positive,0.973837


In [ ]:
nlp_df2.to_excel("amostra_sentimentos_modelonovo.xlsx", index=False)